In [ ]:
# ============================================================
# 0) GOOGLE DRIVE BAĞLANTISI
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 1) GOOGLE COLAB KURULUM
# ============================================================

!pip install scipy pandas numpy matplotlib scikit-learn tqdm -q

import os
import re
import math
import warnings
import requests
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.signal import savgol_filter
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")

BASE_DIR = "/content/oxford_battery"
os.makedirs(BASE_DIR, exist_ok=True)

MAT_PATH = os.path.join(BASE_DIR, "Oxford_Battery_Degradation_Dataset_1.mat")
README_PATH = os.path.join(BASE_DIR, "Readme.txt")

print("Çalışma klasörü:", BASE_DIR)

In [ ]:
# ============================================================
# 2) VERİ SETİNİ İNDİRME
# ============================================================
# Oxford ORA doğrudan dosya linkleri.
# Eğer indirme hata verirse, aşağıdaki manuel yükleme hücresini kullan.

DATA_URL = "https://ora.ox.ac.uk/objects/uuid%3A03ba4b01-cfed-46d3-9b1a-7d4a7bdf6fac/files/m5ac36a1e2073852e4f1f7dee647909a7"
README_URL = "https://ora.ox.ac.uk/objects/uuid%3A03ba4b01-cfed-46d3-9b1a-7d4a7bdf6fac/files/m43cc05e7c5f1245f4895d9dbd495e52f"

def download_file(url, out_path, chunk_size=1024*1024):
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1024:
        print(f"Dosya zaten var: {out_path}")
        return

    print(f"İndiriliyor: {out_path}")
    r = requests.get(url, stream=True)
    r.raise_for_status()

    total = int(r.headers.get("content-length", 0))
    with open(out_path, "wb") as f, tqdm(
        total=total, unit="B", unit_scale=True, desc=os.path.basename(out_path)
    ) as pbar:
        for chunk in r.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))

try:
    download_file(README_URL, README_PATH)
    download_file(DATA_URL, MAT_PATH)
except Exception as e:
    print("Otomatik indirme başarısız olabilir.")
    print("Hata:", e)
    print("\nManuel çözüm:")
    print("1) https://ora.ox.ac.uk/objects/uuid:03ba4b01-cfed-46d3-9b1a-7d4a7bdf6fac adresinden")
    print("2) Oxford_Battery_Degradation_Dataset_1.mat dosyasını indir.")
    print("3) Aşağıdaki manuel yükleme hücresini çalıştır.")

In [ ]:
# ============================================================
# 3) OTOMATİK İNDİRME ÇALIŞMAZSA MANUEL YÜKLEME
# ============================================================
# Bu hücreyi sadece indirme başarısız olursa çalıştır.

# from google.colab import files
# uploaded = files.upload()
# for fname in uploaded.keys():
#     if fname.endswith(".mat"):
#         os.rename(fname, MAT_PATH)
#         print("Yüklendi:", MAT_PATH)

In [ ]:
# ============================================================
# 4) MAT DOSYASINI DAHA SAĞLAM ŞEKİLDE OKU
# ============================================================

import scipy.io as sio
import numpy as np
import re
from scipy.io.matlab import mat_struct

print("MAT dosyası tekrar yükleniyor...")

mat = sio.loadmat(
    MAT_PATH,
    squeeze_me=True,
    struct_as_record=False
)

print("Yüklendi.")
print("Ana değişkenler:")
print([k for k in mat.keys() if not k.startswith("__")])

In [ ]:
# ============================================================
# 5) MATLAB STRUCT -> PYTHON DICT DÖNÜŞÜMÜ
# ============================================================

def matlab_to_python(obj):
    """
    MATLAB mat_struct / ndarray / object array yapılarını
    iç içe Python dict ve numpy array yapısına çevirir.
    """

    # MATLAB struct ise
    if isinstance(obj, mat_struct):
        result = {}
        for field_name in obj._fieldnames:
            result[field_name] = matlab_to_python(getattr(obj, field_name))
        return result

    # Numpy array ise
    if isinstance(obj, np.ndarray):

        # Object array ise içeriğini de dönüştür
        if obj.dtype == object:
            if obj.size == 1:
                return matlab_to_python(obj.item())
            else:
                return np.array([matlab_to_python(x) for x in obj.flat], dtype=object).reshape(obj.shape)

        # Structured array ise
        if obj.dtype.names is not None:
            if obj.size == 1:
                item = obj.item()
                return {name: matlab_to_python(item[name]) for name in obj.dtype.names}
            else:
                return [matlab_to_python(x) for x in obj]

        # Sayısal array ise sıkıştır
        return np.array(obj).squeeze()

    # Diğer her şey
    return obj


data = {}

for cell_idx in range(1, 9):
    cell_name = f"Cell{cell_idx}"

    if cell_name in mat:
        data[cell_name] = matlab_to_python(mat[cell_name])
        print(cell_name, "okundu. Çevrim sayısı:", len(data[cell_name].keys()))
    else:
        print(cell_name, "bulunamadı.")

In [ ]:
# ============================================================
# 6) VERİ YAPISINI HIZLI KONTROL
# ============================================================

def cycle_to_number(cycle_name):
    m = re.search(r"cyc(\d+)", cycle_name)
    if m:
        return int(m.group(1))
    return np.nan


cell_name = "Cell1"

cycle_names = sorted(data[cell_name].keys(), key=cycle_to_number)

print("Örnek çevrim isimleri:", cycle_names[:5])
print("Son çevrim isimleri:", cycle_names[-5:])

sample_cycle = cycle_names[0]

print("\nÖrnek çevrim:", sample_cycle)
print("Tip:", type(data[cell_name][sample_cycle]))

print("Alt alanlar:", data[cell_name][sample_cycle].keys())

print("\nC1ch alanları:", data[cell_name][sample_cycle]["C1ch"].keys())
print("C1dc alanları:", data[cell_name][sample_cycle]["C1dc"].keys())

In [ ]:
# ============================================================
# 7) YARDIMCI FONKSİYONLAR
# ============================================================

def cycle_to_number(cycle_name):
    """
    cyc0000 -> 0
    cyc0100 -> 100
    cyc8000 -> 8000
    """
    m = re.search(r"cyc(\d+)", cycle_name)
    if m:
        return int(m.group(1))
    return np.nan


def to_1d_float(arr):
    """
    MATLAB'dan gelen veriyi temiz 1D float array'e çevirir.
    """
    arr = np.array(arr).squeeze().astype(float)
    arr = arr[np.isfinite(arr)]
    return arr


def get_channel(cycle_dict, mode, key):
    """
    Örnek:
    get_channel(data["Cell1"]["cyc0100"], "C1ch", "v")
    """
    try:
        return to_1d_float(cycle_dict[mode][key])
    except Exception:
        return np.array([])


def compute_capacity_from_discharge(cycle_dict):
    """
    C1dc discharge q verisinden kapasite hesabı.
    q mAh cinsinden geliyor.
    Kapasiteyi q aralığı olarak alıyoruz.
    """
    q = get_channel(cycle_dict, "C1dc", "q")

    if len(q) < 10:
        return np.nan

    capacity = np.nanmax(q) - np.nanmin(q)
    capacity = abs(capacity)

    return capacity


def compute_ica_from_charge(cycle_dict,
                            n_grid=800,
                            v_min_limit=3.3,
                            v_max_limit=4.2,
                            smooth_window=31,
                            smooth_poly=3):
    """
    C1ch charge verisinden ICA eğrisi üretir.
    ICA = dQ / dV

    Çıktılar:
    v_grid : voltaj grid
    ic     : dQ/dV eğrisi
    q_grid : voltaja göre interpolate edilmiş kapasite
    """
    v = get_channel(cycle_dict, "C1ch", "v")
    q = get_channel(cycle_dict, "C1ch", "q")

    if len(v) < 50 or len(q) < 50:
        return None, None, None

    n = min(len(v), len(q))
    v = v[:n]
    q = q[:n]

    mask = np.isfinite(v) & np.isfinite(q)
    v = v[mask]
    q = q[mask]

    # Voltaj aralığını sınırla
    mask = (v >= v_min_limit) & (v <= v_max_limit)
    v = v[mask]
    q = q[mask]

    if len(v) < 50:
        return None, None, None

    # Voltaja göre sırala
    order = np.argsort(v)
    v = v[order]
    q = q[order]

    # Aynı veya çok yakın voltaj değerlerini grupla
    df_tmp = pd.DataFrame({"v": v, "q": q})
    df_tmp["v_round"] = df_tmp["v"].round(4)
    df_g = df_tmp.groupby("v_round", as_index=False)["q"].mean()
    v_unique = df_g["v_round"].values.astype(float)
    q_unique = df_g["q"].values.astype(float)

    if len(v_unique) < 50:
        return None, None, None

    v_grid = np.linspace(v_unique.min(), v_unique.max(), n_grid)
    q_grid = np.interp(v_grid, v_unique, q_unique)

    # Smoothing
    if smooth_window >= len(q_grid):
        smooth_window = len(q_grid) - 1
    if smooth_window % 2 == 0:
        smooth_window -= 1
    if smooth_window < 7:
        smooth_window = 7

    try:
        q_smooth = savgol_filter(q_grid, smooth_window, smooth_poly)
    except Exception:
        q_smooth = q_grid

    ic = np.gradient(q_smooth, v_grid)

    # ICA eğrisini de yumuşat
    try:
        ic = savgol_filter(ic, smooth_window, smooth_poly)
    except Exception:
        pass

    # Negatif ve uç değerleri bastır
    ic[~np.isfinite(ic)] = np.nan

    return v_grid, ic, q_grid


def extract_pmax(cycle_dict, p_window=(3.6, 4.15)):
    """
    ICA eğrisinden Pmax ve Pmax'in voltaj konumunu çıkarır.
    Dataset 2 için ana tepe çoğunlukla 3.8 V civarında olduğundan
    geniş pencere kullanıyoruz.
    """
    v_grid, ic, q_grid = compute_ica_from_charge(cycle_dict)

    if v_grid is None:
        return np.nan, np.nan, None, None

    mask = (
        np.isfinite(ic) &
        (v_grid >= p_window[0]) &
        (v_grid <= p_window[1]) &
        (ic > 0)
    )

    if mask.sum() < 5:
        return np.nan, np.nan, v_grid, ic

    v_sel = v_grid[mask]
    ic_sel = ic[mask]

    # Çok uç gürültüleri azaltmak için yüzde 99.5 üzerini kırpıyoruz
    upper = np.nanpercentile(ic_sel, 99.5)
    ic_sel_clip = np.clip(ic_sel, None, upper)

    idx = np.nanargmax(ic_sel_clip)
    pmax = ic_sel_clip[idx]
    pmax_voltage = v_sel[idx]

    return pmax, pmax_voltage, v_grid, ic

In [ ]:
# ============================================================
# 8) TÜM BATARYALAR İÇİN FEATURE TABLOSU OLUŞTUR
# ============================================================

rows = []
ica_examples = {}

for cell_name in sorted(data.keys()):
    cycles = sorted(data[cell_name].keys(), key=cycle_to_number)

    for cyc_name in tqdm(cycles, desc=cell_name):
        cyc = data[cell_name][cyc_name]
        cycle_number = cycle_to_number(cyc_name)

        capacity_mAh = compute_capacity_from_discharge(cyc)
        pmax, pmax_voltage, v_grid, ic = extract_pmax(cyc)

        # Sıcaklık bilgisi
        T_ch = get_channel(cyc, "C1ch", "T")
        temp_mean = np.nanmean(T_ch) if len(T_ch) > 0 else np.nan
        temp_max = np.nanmax(T_ch) if len(T_ch) > 0 else np.nan

        rows.append({
            "cell": cell_name,
            "cycle_name": cyc_name,
            "cycle_number": cycle_number,
            "capacity_mAh": capacity_mAh,
            "pmax": pmax,
            "pmax_voltage": pmax_voltage,
            "temp_mean": temp_mean,
            "temp_max": temp_max
        })

        # Birkaç ICA örneğini sakla
        if cell_name == "Cell1" and cycle_number in [0, 1000, 2000, 4000, 6000, 8000]:
            if v_grid is not None:
                ica_examples[cyc_name] = (v_grid, ic)

features = pd.DataFrame(rows)

# Geçerli kapasite ve Pmax değerleri
features = features.sort_values(["cell", "cycle_number"]).reset_index(drop=True)

# SoH ve normalized Pmax hesapla
features["soh"] = np.nan
features["pmax_norm"] = np.nan

for cell_name, g in features.groupby("cell"):
    idx = g.index

    first_capacity = g["capacity_mAh"].dropna().iloc[0]
    first_pmax = g["pmax"].dropna().iloc[0]

    features.loc[idx, "soh"] = features.loc[idx, "capacity_mAh"] / first_capacity
    features.loc[idx, "pmax_norm"] = features.loc[idx, "pmax"] / first_pmax

# Temiz tablo
features_clean = features.dropna(subset=["soh", "pmax_norm", "cycle_number"]).copy()

print("Toplam kayıt:", len(features))
print("Geçerli kayıt:", len(features_clean))

features_clean.head()

In [ ]:
# ============================================================
# 9) ÖZET TABLO
# ============================================================

summary = features_clean.groupby("cell").agg(
    n_cycles=("cycle_number", "count"),
    first_cycle=("cycle_number", "min"),
    last_cycle=("cycle_number", "max"),
    first_capacity=("capacity_mAh", "first"),
    last_capacity=("capacity_mAh", "last"),
    first_soh=("soh", "first"),
    last_soh=("soh", "last"),
    first_pmax_norm=("pmax_norm", "first"),
    last_pmax_norm=("pmax_norm", "last"),
).reset_index()

summary

In [ ]:
# ============================================================
# 10) SOH - CYCLE GRAFİĞİ
# ============================================================

plt.figure(figsize=(10, 6))

for cell_name, g in features_clean.groupby("cell"):
    plt.plot(g["cycle_number"], g["soh"], marker="o", label=cell_name)

plt.xlabel("Cycle number")
plt.ylabel("SoH")
plt.title("Oxford Dataset - SoH degradation")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# ============================================================
# 11) ICA EĞRİ ÖRNEKLERİ
# ============================================================

plt.figure(figsize=(10, 6))

for cyc_name, (v_grid, ic) in ica_examples.items():
    plt.plot(v_grid, ic, label=cyc_name)

plt.xlabel("Voltage [V]")
plt.ylabel("dQ/dV [mAh/V]")
plt.title("Cell1 - ICA curves from C1 charge data")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# ============================================================
# 12) NORMALIZED PMAX - SOH İLİŞKİSİ
# ============================================================

plt.figure(figsize=(8, 6))

for cell_name, g in features_clean.groupby("cell"):
    plt.scatter(g["soh"], g["pmax_norm"], label=cell_name, alpha=0.8)

plt.xlabel("SoH")
plt.ylabel("Normalized Pmax")
plt.title("SoH vs Normalized Pmax")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# ============================================================
# 13) KORELASYON ANALİZİ
# ============================================================

corr_all = features_clean[["soh", "pmax_norm", "pmax_voltage", "cycle_number", "temp_mean"]].corr()
print("Genel korelasyon matrisi:")
display(corr_all)

print("\nHücre bazlı SoH - normalized Pmax korelasyonu:")
for cell_name, g in features_clean.groupby("cell"):
    corr = g["soh"].corr(g["pmax_norm"])
    print(f"{cell_name}: {corr:.4f}")

In [ ]:
# ============================================================
# 14) LEAVE-ONE-CELL-OUT MODEL TESTİ
# ============================================================

def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

model_df = features_clean.dropna(subset=[
    "cycle_number", "pmax_norm", "pmax_voltage", "temp_mean", "soh"
]).copy()

feature_cols = ["cycle_number", "pmax_norm", "pmax_voltage", "temp_mean"]
target_col = "soh"

results = []

for test_cell in sorted(model_df["cell"].unique()):
    train_df = model_df[model_df["cell"] != test_cell].copy()
    test_df = model_df[model_df["cell"] == test_cell].copy()

    X_train = train_df[feature_cols].values
    y_train = train_df[target_col].values

    X_test = test_df[feature_cols].values
    y_test = test_df[target_col].values

    # Random Forest
    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2
    )
    rf.fit(X_train, y_train)
    pred_rf = rf.predict(X_test)

    results.append({
        "test_cell": test_cell,
        "model": "RandomForest",
        "RMSE": rmse(y_test, pred_rf),
        "MAE": mean_absolute_error(y_test, pred_rf),
        "R2": r2_score(y_test, pred_rf)
    })

    # Gaussian Process Regression
    kernel = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-4)

    gpr = make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            random_state=42,
            n_restarts_optimizer=3
        )
    )

    gpr.fit(X_train, y_train)
    pred_gpr = gpr.predict(X_test)

    results.append({
        "test_cell": test_cell,
        "model": "GPR",
        "RMSE": rmse(y_test, pred_gpr),
        "MAE": mean_absolute_error(y_test, pred_gpr),
        "R2": r2_score(y_test, pred_gpr)
    })

results_df = pd.DataFrame(results)
display(results_df)

print("\nOrtalama sonuçlar:")
display(results_df.groupby("model")[["RMSE", "MAE", "R2"]].mean().reset_index())

In [ ]:
# ============================================================
# 15) BİR TEST HÜCRESİ İÇİN TAHMİN GRAFİĞİ
# ============================================================

test_cell = "Cell1"

train_df = model_df[model_df["cell"] != test_cell].copy()
test_df = model_df[model_df["cell"] == test_cell].copy()

X_train = train_df[feature_cols].values
y_train = train_df[target_col].values
X_test = test_df[feature_cols].values
y_test = test_df[target_col].values

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    min_samples_leaf=2
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

kernel = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-4)
gpr = make_pipeline(
    StandardScaler(),
    GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=42,
        n_restarts_optimizer=3
    )
)
gpr.fit(X_train, y_train)
pred_gpr = gpr.predict(X_test)

plt.figure(figsize=(10, 6))
plt.plot(test_df["cycle_number"], y_test, marker="o", label="True SoH")
plt.plot(test_df["cycle_number"], pred_rf, marker="s", label="Random Forest")
plt.plot(test_df["cycle_number"], pred_gpr, marker="^", label="GPR")
plt.xlabel("Cycle number")
plt.ylabel("SoH")
plt.title(f"SoH prediction - {test_cell}")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("RF RMSE:", rmse(y_test, pred_rf))
print("GPR RMSE:", rmse(y_test, pred_gpr))

In [ ]:
# ============================================================
# 16) POLYNOMIAL-EKF YARDIMCI FONKSİYONLARI
# ============================================================

def fit_poly_models(train_df):
    """
    Prediction model:
        SoH = f(cycle_number), 2. derece polinom

    Measurement model:
        pmax_norm = h(SoH), 3. derece polinom
    """
    u = train_df["cycle_number"].values.astype(float)
    soh = train_df["soh"].values.astype(float)
    z = train_df["pmax_norm"].values.astype(float)

    pred_coef = np.polyfit(u, soh, deg=2)
    meas_coef = np.polyfit(soh, z, deg=3)

    pred_res = soh - np.polyval(pred_coef, u)
    meas_res = z - np.polyval(meas_coef, soh)

    q = np.var(pred_res)
    r = np.var(meas_res)

    q = max(q, 1e-6)
    r = max(r, 1e-6)

    return pred_coef, meas_coef, q, r


def poly_derivative_value(coef, x):
    """
    Polinom türevi.
    """
    dcoef = np.polyder(coef)
    return np.polyval(dcoef, x)


def polynomial_ekf_predict_update(test_df, pred_coef, meas_coef, q, r, P0=0.01):
    """
    Basit EKF:
    prediction: cycle_number -> SoH
    update: pmax_norm ölçümüyle düzeltme
    """
    test_df = test_df.sort_values("cycle_number").copy()

    xs = []
    Ps = []

    # İlk tahmin
    x = float(np.polyval(pred_coef, test_df.iloc[0]["cycle_number"]))
    P = P0

    for _, row in test_df.iterrows():
        u = float(row["cycle_number"])
        z = float(row["pmax_norm"])

        # Prediction
        x_pred = float(np.polyval(pred_coef, u))
        P_pred = P + q

        # Measurement prediction
        z_pred = float(np.polyval(meas_coef, x_pred))

        # H = dh/dx
        H = float(poly_derivative_value(meas_coef, x_pred))

        # Kalman gain
        S = H * P_pred * H + r
        K = P_pred * H / S

        # Update
        x = x_pred + K * (z - z_pred)
        P = (1 - K * H) * P_pred

        # SoH fiziksel sınırlar
        x = float(np.clip(x, 0.5, 1.05))

        xs.append(x)
        Ps.append(P)

    return np.array(xs), np.array(Ps)

In [ ]:
# ============================================================
# 17) LEAVE-ONE-CELL-OUT POLYNOMIAL-EKF TESTİ
# ============================================================

ekf_results = []

for test_cell in sorted(model_df["cell"].unique()):
    train_df = model_df[model_df["cell"] != test_cell].copy()
    test_df = model_df[model_df["cell"] == test_cell].copy()

    pred_coef, meas_coef, q, r = fit_poly_models(train_df)
    pred_ekf, P_ekf = polynomial_ekf_predict_update(test_df, pred_coef, meas_coef, q, r)

    y_true = test_df["soh"].values

    ekf_results.append({
        "test_cell": test_cell,
        "model": "Polynomial-EKF",
        "RMSE": rmse(y_true, pred_ekf),
        "MAE": mean_absolute_error(y_true, pred_ekf),
        "R2": r2_score(y_true, pred_ekf)
    })

ekf_results_df = pd.DataFrame(ekf_results)
display(ekf_results_df)

print("\nOrtalama Polynomial-EKF:")
display(ekf_results_df.groupby("model")[["RMSE", "MAE", "R2"]].mean().reset_index())

In [ ]:
# ============================================================
# 18) GPR-EKF YARDIMCI FONKSİYONLARI
# ============================================================

def fit_gpr_ekf_models(train_df):
    """
    GPR prediction:
        cycle_number -> SoH

    GPR measurement:
        SoH -> pmax_norm
    """
    kernel_pred = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-4)
    kernel_meas = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-4)

    pred_gpr = make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=kernel_pred,
            normalize_y=True,
            random_state=42,
            n_restarts_optimizer=3
        )
    )

    meas_gpr = make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=kernel_meas,
            normalize_y=True,
            random_state=42,
            n_restarts_optimizer=3
        )
    )

    X_pred = train_df[["cycle_number"]].values.astype(float)
    y_pred = train_df["soh"].values.astype(float)

    X_meas = train_df[["soh"]].values.astype(float)
    y_meas = train_df["pmax_norm"].values.astype(float)

    pred_gpr.fit(X_pred, y_pred)
    meas_gpr.fit(X_meas, y_meas)

    return pred_gpr, meas_gpr


def predict_with_std(model, X):
    """
    Pipeline içindeki GPR'den std almak için yardımcı fonksiyon.
    """
    scaler = model.named_steps["standardscaler"]
    gpr = model.named_steps["gaussianprocessregressor"]

    X_scaled = scaler.transform(X)
    mean, std = gpr.predict(X_scaled, return_std=True)
    return mean, std


def gpr_model_predict(model, x_value):
    X = np.array([[x_value]], dtype=float)
    mean, std = predict_with_std(model, X)
    return float(mean[0]), float(std[0])


def numerical_derivative_gpr(model, x, delta=1e-4):
    """
    h'(x) sayısal türev.
    """
    y_plus, _ = gpr_model_predict(model, x + delta)
    y_minus, _ = gpr_model_predict(model, x - delta)
    return (y_plus - y_minus) / (2 * delta)


def gpr_ekf_predict_update(test_df, pred_gpr, meas_gpr, P0=0.01):
    test_df = test_df.sort_values("cycle_number").copy()

    xs = []
    Ps = []

    x, std0 = gpr_model_predict(pred_gpr, float(test_df.iloc[0]["cycle_number"]))
    P = P0

    for _, row in test_df.iterrows():
        u = float(row["cycle_number"])
        z = float(row["pmax_norm"])

        # Prediction: GPR cycle -> SoH
        x_pred, pred_std = gpr_model_predict(pred_gpr, u)
        q = max(pred_std ** 2, 1e-6)
        P_pred = P + q

        # Measurement: GPR SoH -> pmax_norm
        z_pred, meas_std = gpr_model_predict(meas_gpr, x_pred)
        r = max(meas_std ** 2, 1e-6)

        # H = dh/dx
        H = numerical_derivative_gpr(meas_gpr, x_pred, delta=1e-4)

        # Çok küçük H durumunda güncelleme sorunlu olabilir
        if abs(H) < 1e-8:
            x = x_pred
            P = P_pred
        else:
            S = H * P_pred * H + r
            K = P_pred * H / S

            x = x_pred + K * (z - z_pred)
            P = (1 - K * H) * P_pred

        x = float(np.clip(x, 0.5, 1.05))

        xs.append(x)
        Ps.append(P)

    return np.array(xs), np.array(Ps)

In [ ]:
# ============================================================
# 19) LEAVE-ONE-CELL-OUT GPR-EKF TESTİ
# ============================================================

gpr_ekf_results = []

for test_cell in sorted(model_df["cell"].unique()):
    train_df = model_df[model_df["cell"] != test_cell].copy()
    test_df = model_df[model_df["cell"] == test_cell].copy()

    pred_gpr, meas_gpr = fit_gpr_ekf_models(train_df)
    pred_gpr_ekf, P_gpr_ekf = gpr_ekf_predict_update(test_df, pred_gpr, meas_gpr)

    y_true = test_df["soh"].values

    gpr_ekf_results.append({
        "test_cell": test_cell,
        "model": "GPR-EKF-simple",
        "RMSE": rmse(y_true, pred_gpr_ekf),
        "MAE": mean_absolute_error(y_true, pred_gpr_ekf),
        "R2": r2_score(y_true, pred_gpr_ekf)
    })

gpr_ekf_results_df = pd.DataFrame(gpr_ekf_results)
display(gpr_ekf_results_df)

print("\nOrtalama GPR-EKF-simple:")
display(gpr_ekf_results_df.groupby("model")[["RMSE", "MAE", "R2"]].mean().reset_index())

In [ ]:
# ============================================================
# 20) TÜM SONUÇLARI BİRLEŞTİR
# ============================================================

all_results = pd.concat([
    results_df,
    ekf_results_df,
    gpr_ekf_results_df
], ignore_index=True)

display(all_results)

print("\nModel bazlı ortalama sonuç:")
avg_results = all_results.groupby("model")[["RMSE", "MAE", "R2"]].mean().reset_index()
display(avg_results.sort_values("RMSE"))

In [ ]:
# ============================================================
# 21) BİR BATARYA İÇİN TÜM MODEL GRAFİĞİ
# ============================================================

test_cell = "Cell1"

train_df = model_df[model_df["cell"] != test_cell].copy()
test_df = model_df[model_df["cell"] == test_cell].copy()

X_train = train_df[feature_cols].values
y_train = train_df[target_col].values
X_test = test_df[feature_cols].values
y_test = test_df[target_col].values

# RF
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    min_samples_leaf=2
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

# GPR
kernel = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-4)
gpr = make_pipeline(
    StandardScaler(),
    GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=42,
        n_restarts_optimizer=3
    )
)
gpr.fit(X_train, y_train)
pred_gpr = gpr.predict(X_test)

# Polynomial-EKF
pred_coef, meas_coef, q, r = fit_poly_models(train_df)
pred_poly_ekf, _ = polynomial_ekf_predict_update(test_df, pred_coef, meas_coef, q, r)

# GPR-EKF
pred_gpr_model, meas_gpr_model = fit_gpr_ekf_models(train_df)
pred_gpr_ekf, _ = gpr_ekf_predict_update(test_df, pred_gpr_model, meas_gpr_model)

plt.figure(figsize=(11, 6))
plt.plot(test_df["cycle_number"], y_test, marker="o", linewidth=2, label="True SoH")
plt.plot(test_df["cycle_number"], pred_rf, marker="s", label="Random Forest")
plt.plot(test_df["cycle_number"], pred_gpr, marker="^", label="GPR")
plt.plot(test_df["cycle_number"], pred_poly_ekf, marker="x", label="Polynomial-EKF")
plt.plot(test_df["cycle_number"], pred_gpr_ekf, marker="d", label="GPR-EKF-simple")

plt.xlabel("Cycle number")
plt.ylabel("SoH")
plt.title(f"Model comparison - {test_cell}")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("RMSE değerleri:")
print("RF:", rmse(y_test, pred_rf))
print("GPR:", rmse(y_test, pred_gpr))
print("Polynomial-EKF:", rmse(y_test, pred_poly_ekf))
print("GPR-EKF-simple:", rmse(y_test, pred_gpr_ekf))

In [ ]:
# %%
#
# # 22) Özgün Çalışmaya Geçiş: Multi-HI, Partial Charging, Outlier Analizi ve SHAP
#
# Buradan sonraki hücreler, önceki baseline kodun devamıdır. Amaç artık örnek makaledeki tek `normalized Pmax` yaklaşımından çıkıp daha özgün bir hatta geçmek:
#
# - Tek HI yerine çoklu sağlık göstergeleri çıkarılır.
# - Farklı partial charging voltaj pencereleri denenir.
# - Kapasite dip / outlier noktaları işaretlenir.
# - Leave-one-cell-out protokolüyle modeller karşılaştırılır.
# - SHAP ile hangi göstergelerin SoH tahmininde daha etkili olduğu açıklanır.
#
# Bu bölüm önceki hücrelerde oluşan `data`, `features_clean`, `get_channel`, `cycle_to_number`, `rmse` gibi değişken ve fonksiyonların hazır olduğunu varsayar.

# %%

# ============================================================
# 22) MULTI-HI VE PARTIAL CHARGING İÇİN YARDIMCI FONKSİYONLAR
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import os
import subprocess
import sys
from tqdm.auto import tqdm

from scipy.signal import savgol_filter
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.inspection import permutation_importance

# Partial charging senaryoları
# Full: makaleye yakın, geniş pencere
# Medium/Narrow/VeryNarrow: gerçek kullanımda tam şarj yokmuş gibi daha dar pencereler
PARTIAL_WINDOWS = {
    "full_36_42": (3.60, 4.20),
    "medium_37_40": (3.70, 4.00),
    "narrow_375_395": (3.75, 3.95),
    "very_narrow_38_39": (3.80, 3.90),
}

def safe_get_channel(cycle_dict, mode, key_candidates):
    """
    key_candidates tek string veya liste olabilir.
    Örn: safe_get_channel(cyc, "C1ch", ["t", "time", "Time"])
    """
    if isinstance(key_candidates, str):
        key_candidates = [key_candidates]

    for key in key_candidates:
        try:
            arr = get_channel(cycle_dict, mode, key)
            if arr is not None and len(arr) > 0:
                return arr
        except Exception:
            pass

    return np.array([])


def get_charge_arrays(cycle_dict):
    """
    C1ch alanından voltaj, kapasite, zaman ve sıcaklık dizilerini alır.
    """
    v = safe_get_channel(cycle_dict, "C1ch", ["v", "V", "voltage", "Voltage"])
    q = safe_get_channel(cycle_dict, "C1ch", ["q", "Q", "capacity", "Capacity"])
    t = safe_get_channel(cycle_dict, "C1ch", ["t", "Tt", "time", "Time"])
    T = safe_get_channel(cycle_dict, "C1ch", ["T", "temp", "Temp", "temperature", "Temperature"])

    n = min([len(x) for x in [v, q] if len(x) > 0]) if len(v) > 0 and len(q) > 0 else 0

    if n == 0:
        return np.array([]), np.array([]), np.array([]), np.array([])

    v = np.asarray(v[:n], dtype=float)
    q = np.asarray(q[:n], dtype=float)

    if len(t) >= n:
        t = np.asarray(t[:n], dtype=float)
    else:
        t = np.full(n, np.nan)

    if len(T) >= n:
        T = np.asarray(T[:n], dtype=float)
    else:
        T = np.full(n, np.nan)

    mask = np.isfinite(v) & np.isfinite(q)
    return v[mask], q[mask], t[mask], T[mask]


def compute_ica_and_dva_window(cycle_dict, v_window, n_grid=500, smooth_window=31, smooth_poly=3):
    """
    Verilen voltaj penceresinde ICA ve DVA türetir.

    ICA = dQ/dV
    DVA = dV/dQ

    Çıktılar:
    v_grid, q_grid, ic, dva, raw_window_df
    """
    v, q, t, T = get_charge_arrays(cycle_dict)

    if len(v) < 30:
        return None, None, None, None, None

    lo, hi = v_window
    mask = np.isfinite(v) & np.isfinite(q) & (v >= lo) & (v <= hi)

    if mask.sum() < 30:
        return None, None, None, None, None

    v_w = v[mask]
    q_w = q[mask]
    t_w = t[mask] if len(t) == len(v) else np.full(mask.sum(), np.nan)
    T_w = T[mask] if len(T) == len(v) else np.full(mask.sum(), np.nan)

    # Voltaja göre sırala ve aynı voltajları grupla
    df_tmp = pd.DataFrame({"v": v_w, "q": q_w, "t": t_w, "T": T_w})
    df_tmp = df_tmp.replace([np.inf, -np.inf], np.nan).dropna(subset=["v", "q"])

    if len(df_tmp) < 30:
        return None, None, None, None, None

    df_tmp["v_round"] = df_tmp["v"].round(4)
    df_g = df_tmp.groupby("v_round", as_index=False).agg(
        q=("q", "mean"),
        t=("t", "mean"),
        T=("T", "mean")
    )
    df_g = df_g.rename(columns={"v_round": "v"}).sort_values("v")

    v_unique = df_g["v"].values.astype(float)
    q_unique = df_g["q"].values.astype(float)

    if len(v_unique) < 20:
        return None, None, None, None, None

    # Grid
    v_grid = np.linspace(v_unique.min(), v_unique.max(), n_grid)
    q_grid = np.interp(v_grid, v_unique, q_unique)

    # Smoothing penceresini ayarla
    sw = smooth_window
    if sw >= len(q_grid):
        sw = len(q_grid) - 1
    if sw % 2 == 0:
        sw -= 1
    if sw < 7:
        sw = 7

    try:
        q_smooth = savgol_filter(q_grid, sw, smooth_poly)
    except Exception:
        q_smooth = q_grid

    # ICA
    ic = np.gradient(q_smooth, v_grid)

    # DVA için q tekrarlı / düz gidebilir, güvenli türev alalım
    dq = np.gradient(q_smooth)
    dv = np.gradient(v_grid)
    with np.errstate(divide="ignore", invalid="ignore"):
        dva = dv / dq

    try:
        ic = savgol_filter(ic, sw, smooth_poly)
    except Exception:
        pass

    ic[~np.isfinite(ic)] = np.nan
    dva[~np.isfinite(dva)] = np.nan

    return v_grid, q_grid, ic, dva, df_tmp


def peak_width_half_max(v_grid, ic):
    """
    ICA tepesinin yarı maksimum genişliği.
    """
    if v_grid is None or ic is None:
        return np.nan

    mask = np.isfinite(ic) & (ic > 0)
    if mask.sum() < 5:
        return np.nan

    v = v_grid[mask]
    y = ic[mask]

    y_max = np.nanmax(y)
    if not np.isfinite(y_max) or y_max <= 0:
        return np.nan

    half = 0.5 * y_max
    above = y >= half

    if above.sum() < 2:
        return np.nan

    return float(v[above].max() - v[above].min())


def extract_multi_hi_for_window(cycle_dict, v_window):
    """
    Bir çevrim ve bir voltaj penceresi için çoklu HI çıkarır.
    """
    v_grid, q_grid, ic, dva, raw_df = compute_ica_and_dva_window(cycle_dict, v_window)

    out = {
        "pmax": np.nan,
        "pmax_voltage": np.nan,
        "ic_area": np.nan,
        "ic_mean": np.nan,
        "ic_std": np.nan,
        "peak_width": np.nan,
        "q_delta_window": np.nan,
        "charge_time_window": np.nan,
        "dva_mean": np.nan,
        "dva_std": np.nan,
        "dva_abs_mean": np.nan,
        "temp_mean_window": np.nan,
        "temp_max_window": np.nan,
    }

    if v_grid is None:
        return out

    mask_ic = np.isfinite(ic) & (ic > 0)
    if mask_ic.sum() >= 5:
        ic_pos = ic[mask_ic]
        v_pos = v_grid[mask_ic]

        # Aşırı uçları bastır
        upper = np.nanpercentile(ic_pos, 99.5)
        ic_clip = np.clip(ic_pos, None, upper)

        idx = int(np.nanargmax(ic_clip))
        out["pmax"] = float(ic_clip[idx])
        out["pmax_voltage"] = float(v_pos[idx])
        out["ic_area"] = float(np.trapz(ic_clip, v_pos))
        out["ic_mean"] = float(np.nanmean(ic_clip))
        out["ic_std"] = float(np.nanstd(ic_clip))
        out["peak_width"] = peak_width_half_max(v_grid, ic)

    if q_grid is not None and np.isfinite(q_grid).sum() > 5:
        out["q_delta_window"] = float(np.nanmax(q_grid) - np.nanmin(q_grid))

    if raw_df is not None and len(raw_df) > 0:
        if "t" in raw_df.columns and raw_df["t"].notna().sum() > 2:
            out["charge_time_window"] = float(raw_df["t"].max() - raw_df["t"].min())

        if "T" in raw_df.columns and raw_df["T"].notna().sum() > 2:
            out["temp_mean_window"] = float(raw_df["T"].mean())
            out["temp_max_window"] = float(raw_df["T"].max())

    if dva is not None:
        mask_dva = np.isfinite(dva)
        if mask_dva.sum() > 5:
            dva_sel = dva[mask_dva]
            # DVA bazen uç değer verebilir, kırp
            lo, hi = np.nanpercentile(dva_sel, [1, 99])
            dva_clip = np.clip(dva_sel, lo, hi)
            out["dva_mean"] = float(np.nanmean(dva_clip))
            out["dva_std"] = float(np.nanstd(dva_clip))
            out["dva_abs_mean"] = float(np.nanmean(np.abs(dva_clip)))

    return out


def detect_soh_outliers(df, threshold=0.04):
    """
    Kapasite dip / ani SoH sapmalarını işaretler.
    Silmez, sadece flag üretir.
    """
    df = df.sort_values(["cell", "cycle_number"]).copy()
    df["soh_rolling_median"] = np.nan
    df["soh_outlier"] = False
    df["soh_delta"] = np.nan

    for cell_name, g in df.groupby("cell"):
        idx = g.index
        s = g["soh"].astype(float)
        med = s.rolling(window=5, center=True, min_periods=2).median()
        delta = s.diff()

        outlier = (np.abs(s - med) > threshold) | (delta < -threshold)

        df.loc[idx, "soh_rolling_median"] = med.values
        df.loc[idx, "soh_delta"] = delta.values
        df.loc[idx, "soh_outlier"] = outlier.fillna(False).values

    return df


print("Multi-HI yardımcı fonksiyonları hazır.")
print("Partial charging pencereleri:", PARTIAL_WINDOWS)

In [ ]:
# ============================================================
# 23) TÜM BATARYALAR İÇİN MULTI-HI FEATURE TABLOSU OLUŞTUR
# ============================================================

multi_rows = []

for cell_name in sorted(data.keys()):
    cycles = sorted(data[cell_name].keys(), key=cycle_to_number)

    for cyc_name in tqdm(cycles, desc=f"Multi-HI {cell_name}"):
        cyc = data[cell_name][cyc_name]
        cycle_number = cycle_to_number(cyc_name)

        # Önceki kodda tanımlı kapasite fonksiyonunu kullanıyoruz
        capacity_mAh = compute_capacity_from_discharge(cyc)

        # Genel sıcaklık
        T_ch = safe_get_channel(cyc, "C1ch", ["T", "temp", "Temp", "temperature", "Temperature"])
        temp_mean = np.nanmean(T_ch) if len(T_ch) > 0 else np.nan
        temp_max = np.nanmax(T_ch) if len(T_ch) > 0 else np.nan

        row = {
            "cell": cell_name,
            "cycle_name": cyc_name,
            "cycle_number": cycle_number,
            "capacity_mAh": capacity_mAh,
            "temp_mean": temp_mean,
            "temp_max": temp_max,
        }

        # Her partial window için özellik çıkar
        for tag, win in PARTIAL_WINDOWS.items():
            feats = extract_multi_hi_for_window(cyc, win)
            for k, v in feats.items():
                row[f"{k}_{tag}"] = v

        multi_rows.append(row)

multi_features = pd.DataFrame(multi_rows)
multi_features = multi_features.sort_values(["cell", "cycle_number"]).reset_index(drop=True)

# SoH ve window bazlı normalized Pmax hesapla
multi_features["soh"] = np.nan

for cell_name, g in multi_features.groupby("cell"):
    idx = g.index

    first_capacity = g["capacity_mAh"].dropna().iloc[0]
    multi_features.loc[idx, "soh"] = multi_features.loc[idx, "capacity_mAh"] / first_capacity

    # Her pencere için Pmax'i kendi ilk geçerli değerine göre normalize et
    for tag in PARTIAL_WINDOWS.keys():
        pcol = f"pmax_{tag}"
        ncol = f"pmax_norm_{tag}"

        if pcol in multi_features.columns and g[pcol].dropna().shape[0] > 0:
            first_pmax = g[pcol].dropna().iloc[0]
            multi_features.loc[idx, ncol] = multi_features.loc[idx, pcol] / first_pmax
        else:
            multi_features.loc[idx, ncol] = np.nan

# Outlier flag
multi_features = detect_soh_outliers(multi_features, threshold=0.04)

print("Multi-HI feature tablosu hazır.")
print("Boyut:", multi_features.shape)
display(multi_features.head())

print("\nPencere bazlı geçerli Pmax sayıları:")
coverage_rows = []
for tag in PARTIAL_WINDOWS.keys():
    n_valid = multi_features[f"pmax_norm_{tag}"].notna().sum()
    coverage_rows.append({
        "window": tag,
        "valid_rows": n_valid,
        "total_rows": len(multi_features),
        "coverage_percent": 100 * n_valid / len(multi_features)
    })
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

In [ ]:
# ============================================================
# 24) MULTI-HI KORELASYON ANALİZİ
# ============================================================

# SoH ile en ilişkili özellikleri bulalım
numeric_cols = multi_features.select_dtypes(include=[np.number]).columns.tolist()

corr_rows = []
for col in numeric_cols:
    if col in ["soh"]:
        continue
    valid = multi_features[["soh", col]].dropna()
    if len(valid) < 10:
        continue
    corr_rows.append({
        "feature": col,
        "corr_with_soh": valid["soh"].corr(valid[col]),
        "abs_corr": abs(valid["soh"].corr(valid[col])),
        "valid_count": len(valid)
    })

corr_multi_df = pd.DataFrame(corr_rows).sort_values("abs_corr", ascending=False).reset_index(drop=True)

print("SoH ile en yüksek mutlak korelasyona sahip ilk 30 özellik:")
display(corr_multi_df.head(30))

# Pencere bazlı normalized Pmax korelasyonları
print("\nPencere bazlı SoH - normalized Pmax korelasyonu:")
for tag in PARTIAL_WINDOWS.keys():
    col = f"pmax_norm_{tag}"
    valid = multi_features[["soh", col]].dropna()
    corr = valid["soh"].corr(valid[col]) if len(valid) > 10 else np.nan
    print(f"{tag}: corr={corr:.4f}, n={len(valid)}")

# Pencere bazlı scatter plot
for tag in PARTIAL_WINDOWS.keys():
    col = f"pmax_norm_{tag}"
    if col not in multi_features.columns:
        continue

    plt.figure(figsize=(7, 5))
    for cell_name, g in multi_features.groupby("cell"):
        plt.scatter(g["soh"], g[col], s=18, alpha=0.75, label=cell_name)

    plt.xlabel("SoH")
    plt.ylabel(f"Normalized Pmax ({tag})")
    plt.title(f"SoH vs Normalized Pmax - {tag}")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.show()


In [ ]:
# ============================================================
# 25) WINDOW BAZLI FEATURE SETLERİ VE MODEL YARDIMCILARI
# ============================================================

def get_feature_cols_for_window(tag):
    """
    Bir partial charging penceresi için kullanılacak çoklu HI özelliklerini döndürür.
    """
    base_cols = [
        "cycle_number",
        "temp_mean",
        "temp_max",
    ]

    window_cols = [
        f"pmax_norm_{tag}",
        f"pmax_voltage_{tag}",
        f"ic_area_{tag}",
        f"ic_mean_{tag}",
        f"ic_std_{tag}",
        f"peak_width_{tag}",
        f"q_delta_window_{tag}",
        f"charge_time_window_{tag}",
        f"dva_mean_{tag}",
        f"dva_std_{tag}",
        f"dva_abs_mean_{tag}",
        f"temp_mean_window_{tag}",
        f"temp_max_window_{tag}",
    ]

    cols = [c for c in base_cols + window_cols if c in multi_features.columns]
    return cols


def make_model(model_name):
    """
    Karşılaştırma için modeller.
    Not: SimpleImputer ile eksik HI değerleri medyanla doldurulur.
    """
    if model_name == "MultiHI-RandomForest":
        return make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestRegressor(
                n_estimators=400,
                random_state=42,
                min_samples_leaf=2,
                n_jobs=-1
            )
        )

    if model_name == "MultiHI-ExtraTrees":
        return make_pipeline(
            SimpleImputer(strategy="median"),
            ExtraTreesRegressor(
                n_estimators=400,
                random_state=42,
                min_samples_leaf=2,
                n_jobs=-1
            )
        )

    if model_name == "MultiHI-HistGB":
        return make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingRegressor(
                random_state=42,
                max_iter=300,
                learning_rate=0.04,
                l2_regularization=0.01
            )
        )

    if model_name == "MultiHI-Ridge":
        return make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            Ridge(alpha=1.0)
        )

    raise ValueError(f"Bilinmeyen model: {model_name}")


def evaluate_leave_one_cell_out(
    df,
    feature_cols,
    target_col="soh",
    model_names=None,
    exclude_outliers=False,
    min_test_rows=5
):
    """
    Leave-one-cell-out değerlendirme.
    """
    if model_names is None:
        model_names = [
            "MultiHI-RandomForest",
            "MultiHI-ExtraTrees",
            "MultiHI-HistGB",
            "MultiHI-Ridge",
        ]

    eval_df = df.copy()

    if exclude_outliers and "soh_outlier" in eval_df.columns:
        eval_df = eval_df[~eval_df["soh_outlier"]].copy()

    # Hedef ve en az bir feature dolu olsun
    needed = ["cell", target_col] + feature_cols
    eval_df = eval_df[needed].copy()
    eval_df = eval_df.dropna(subset=[target_col])

    results = []
    preds_all = []

    for test_cell in sorted(eval_df["cell"].unique()):
        train_df = eval_df[eval_df["cell"] != test_cell].copy()
        test_df = eval_df[eval_df["cell"] == test_cell].copy()

        if len(test_df) < min_test_rows or len(train_df) < 20:
            continue

        X_train = train_df[feature_cols]
        y_train = train_df[target_col].values
        X_test = test_df[feature_cols]
        y_test = test_df[target_col].values

        for model_name in model_names:
            model = make_model(model_name)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            results.append({
                "test_cell": test_cell,
                "model": model_name,
                "RMSE": rmse(y_test, y_pred),
                "MAE": mean_absolute_error(y_test, y_pred),
                "R2": r2_score(y_test, y_pred),
                "n_test": len(test_df),
                "exclude_outliers": exclude_outliers
            })

            tmp = test_df[["cell"]].copy()
            tmp["test_cell"] = test_cell
            tmp["model"] = model_name
            tmp["y_true"] = y_test
            tmp["y_pred"] = y_pred
            preds_all.append(tmp)

    results_df = pd.DataFrame(results)
    preds_df = pd.concat(preds_all, ignore_index=True) if len(preds_all) else pd.DataFrame()

    return results_df, preds_df


print("Model yardımcı fonksiyonları hazır.")
for tag in PARTIAL_WINDOWS.keys():
    print(tag, "feature sayısı:", len(get_feature_cols_for_window(tag)))

In [ ]:
# ============================================================
# 26) PARTIAL CHARGING SENARYOLARINDA MULTI-HI MODEL KARŞILAŞTIRMASI
# ============================================================

partial_results = []
partial_preds = []

for tag in PARTIAL_WINDOWS.keys():
    feature_cols = get_feature_cols_for_window(tag)

    # Çok eksik olan pencereyi yine de deneyelim ama kayıt tutalım
    print("\n" + "="*70)
    print("Window:", tag)
    print("Feature cols:", feature_cols)

    res_raw, preds_raw = evaluate_leave_one_cell_out(
        multi_features,
        feature_cols=feature_cols,
        exclude_outliers=False
    )
    res_raw["window"] = tag
    partial_results.append(res_raw)
    if len(preds_raw):
        preds_raw["window"] = tag
        partial_preds.append(preds_raw)

    res_clean, preds_clean = evaluate_leave_one_cell_out(
        multi_features,
        feature_cols=feature_cols,
        exclude_outliers=True
    )
    res_clean["window"] = tag
    partial_results.append(res_clean)
    if len(preds_clean):
        preds_clean["window"] = tag
        partial_preds.append(preds_clean)

partial_results_df = pd.concat(partial_results, ignore_index=True)
partial_preds_df = pd.concat(partial_preds, ignore_index=True) if len(partial_preds) else pd.DataFrame()

print("\nTüm sonuçlar:")
display(partial_results_df)

print("\nWindow + model bazlı ortalama sonuçlar:")
partial_avg = (
    partial_results_df
    .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
    .mean()
    .reset_index()
    .sort_values(["exclude_outliers", "RMSE"])
)
display(partial_avg)

# En iyi sonuçlar
print("\nEn iyi 15 kombinasyon:")
display(partial_avg.head(15))

In [ ]:
# ============================================================
# 27) SINGLE-HI EKF BASELINE'I WINDOW BAZINDA KARŞILAŞTIR
# ============================================================
# Bu bölüm makaleye daha yakın single-HI yaklaşımını partial window'larda test eder.
# Ölçüm olarak sadece pmax_norm_{window} kullanılır.

def fit_poly_models_generic(train_df, meas_col, cycle_col="cycle_number", target_col="soh"):
    train_df = train_df.dropna(subset=[cycle_col, target_col, meas_col]).copy()

    u = train_df[cycle_col].values.astype(float)
    soh = train_df[target_col].values.astype(float)
    z = train_df[meas_col].values.astype(float)

    pred_coef = np.polyfit(u, soh, deg=2)
    meas_coef = np.polyfit(soh, z, deg=3)

    pred_res = soh - np.polyval(pred_coef, u)
    meas_res = z - np.polyval(meas_coef, soh)

    q = max(np.var(pred_res), 1e-6)
    r = max(np.var(meas_res), 1e-6)

    return pred_coef, meas_coef, q, r


def polynomial_ekf_generic(test_df, pred_coef, meas_coef, q, r, meas_col, P0=0.01):
    test_df = test_df.sort_values("cycle_number").copy()

    xs, Ps = [], []
    P = P0

    # İlk değer
    first_u = float(test_df.iloc[0]["cycle_number"])
    x = float(np.polyval(pred_coef, first_u))

    for _, row in test_df.iterrows():
        u = float(row["cycle_number"])
        z = row[meas_col]

        # Prediction
        x_pred = float(np.polyval(pred_coef, u))
        P_pred = P + q

        if not np.isfinite(z):
            # Ölçüm yoksa sadece prediction
            x = x_pred
            P = P_pred
        else:
            z_pred = float(np.polyval(meas_coef, x_pred))
            H = float(poly_derivative_value(meas_coef, x_pred))
            S = H * P_pred * H + r

            if S <= 0 or abs(H) < 1e-10:
                x = x_pred
                P = P_pred
            else:
                K = P_pred * H / S
                x = x_pred + K * (float(z) - z_pred)
                P = (1 - K * H) * P_pred

        x = float(np.clip(x, 0.5, 1.05))
        xs.append(x)
        Ps.append(P)

    return np.array(xs), np.array(Ps)


single_hi_results = []

for tag in PARTIAL_WINDOWS.keys():
    meas_col = f"pmax_norm_{tag}"

    for exclude_outliers in [False, True]:
        df_eval = multi_features.copy()
        if exclude_outliers:
            df_eval = df_eval[~df_eval["soh_outlier"]].copy()

        for test_cell in sorted(df_eval["cell"].unique()):
            train_df = df_eval[df_eval["cell"] != test_cell].copy()
            test_df = df_eval[df_eval["cell"] == test_cell].copy()

            # Eğitim için ölçüm kolonu dolu olmalı
            train_valid = train_df.dropna(subset=["cycle_number", "soh", meas_col]).copy()
            test_valid = test_df.dropna(subset=["cycle_number", "soh"]).copy()

            if len(train_valid) < 20 or len(test_valid) < 5:
                continue

            pred_coef, meas_coef, q, r = fit_poly_models_generic(train_valid, meas_col=meas_col)
            y_pred, _ = polynomial_ekf_generic(test_valid, pred_coef, meas_coef, q, r, meas_col=meas_col)

            y_true = test_valid["soh"].values

            single_hi_results.append({
                "window": tag,
                "test_cell": test_cell,
                "model": "SingleHI-Polynomial-EKF",
                "RMSE": rmse(y_true, y_pred),
                "MAE": mean_absolute_error(y_true, y_pred),
                "R2": r2_score(y_true, y_pred),
                "n_test": len(test_valid),
                "exclude_outliers": exclude_outliers
            })

single_hi_results_df = pd.DataFrame(single_hi_results)

print("Single-HI Polynomial-EKF sonuçları:")
display(single_hi_results_df)

print("\nSingle-HI window bazlı ortalama:")
single_hi_avg = (
    single_hi_results_df
    .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
    .mean()
    .reset_index()
    .sort_values(["exclude_outliers", "RMSE"])
)
display(single_hi_avg)

In [ ]:
# ============================================================
# 28) SINGLE-HI VS MULTI-HI ÖZET KARŞILAŞTIRMA
# ============================================================

multi_avg_for_compare = (
    partial_results_df
    .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
    .mean()
    .reset_index()
)

summary_compare = pd.concat([
    single_hi_avg,
    multi_avg_for_compare
], ignore_index=True)

summary_compare = summary_compare.sort_values(["exclude_outliers", "RMSE"]).reset_index(drop=True)

print("Single-HI ve Multi-HI tüm yöntemlerin ortalama karşılaştırması:")
display(summary_compare)

# Her window için en iyi yöntem
best_by_window = (
    summary_compare
    .sort_values("RMSE")
    .groupby(["exclude_outliers", "window"], as_index=False)
    .first()
)

print("\nHer partial charging penceresi için en iyi yöntem:")
display(best_by_window)

# Grafik
for exclude_flag in [False, True]:
    tmp = summary_compare[summary_compare["exclude_outliers"] == exclude_flag].copy()

    plt.figure(figsize=(11, 5))
    labels = tmp["window"] + "\n" + tmp["model"].str.replace("MultiHI-", "", regex=False).str.replace("SingleHI-", "", regex=False)
    plt.bar(range(len(tmp)), tmp["RMSE"])
    plt.xticks(range(len(tmp)), labels, rotation=90)
    plt.ylabel("Average RMSE")
    plt.title(f"Single-HI vs Multi-HI comparison | exclude_outliers={exclude_flag}")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 29) MISSING HI / EKSİK ÖLÇÜM DAYANIKLILIK TESTİ
# ============================================================
# Burada belirli oranlarda HI özelliklerini rastgele eksiltiyoruz.
# Ama cycle_number gibi temel bilgiyi silmiyoruz.
# SimpleImputer kullanan Multi-HI modellerin eksik ölçüme ne kadar dayanıklı olduğunu görüyoruz.

def simulate_missing_hi(df, feature_cols, missing_prob=0.2, seed=42):
    rng = np.random.default_rng(seed)
    df_m = df.copy()

    # Silinmeyecek temel kolonlar
    protected = {"cycle_number", "soh", "cell", "cycle_name", "capacity_mAh", "soh_outlier"}
    candidate_cols = [c for c in feature_cols if c not in protected and c in df_m.columns]

    for col in candidate_cols:
        mask = rng.random(len(df_m)) < missing_prob
        df_m.loc[mask, col] = np.nan

    return df_m


missing_results = []

# En iyi veya makul pencereyi seçelim: full ve medium daha güvenilir
missing_windows_to_test = ["full_36_42", "medium_37_40", "narrow_375_395"]
missing_probs = [0.0, 0.2, 0.4, 0.6, 0.8]

for tag in missing_windows_to_test:
    feature_cols = get_feature_cols_for_window(tag)

    for p in missing_probs:
        df_missing = simulate_missing_hi(
            multi_features,
            feature_cols=feature_cols,
            missing_prob=p,
            seed=42
        )

        res, _ = evaluate_leave_one_cell_out(
            df_missing,
            feature_cols=feature_cols,
            model_names=["MultiHI-RandomForest", "MultiHI-ExtraTrees", "MultiHI-HistGB"],
            exclude_outliers=False
        )

        res["window"] = tag
        res["missing_prob"] = p
        missing_results.append(res)

missing_results_df = pd.concat(missing_results, ignore_index=True)

missing_avg = (
    missing_results_df
    .groupby(["window", "missing_prob", "model"])[["RMSE", "MAE", "R2"]]
    .mean()
    .reset_index()
)

print("Eksik HI dayanıklılık sonuçları:")
display(missing_avg)

for tag in missing_windows_to_test:
    tmp = missing_avg[missing_avg["window"] == tag]

    plt.figure(figsize=(8, 5))
    for model_name, g in tmp.groupby("model"):
        plt.plot(g["missing_prob"], g["RMSE"], marker="o", label=model_name)

    plt.xlabel("Missing HI probability")
    plt.ylabel("Average RMSE")
    plt.title(f"Missing HI robustness - {tag}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


In [ ]:
# ============================================================
# 30) SHAP AÇIKLANABİLİRLİK ANALİZİ
# ============================================================
# SHAP, hangi Multi-HI özelliklerinin SoH tahmininde etkili olduğunu gösterir.
# Eğer shap kurulu değilse Colab üzerinde kurar.

try:
    import shap
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "-q"])
    import shap

# Numpy sürüm uyumluluğu için
if not hasattr(np, "bool"):
    np.bool = bool

# SHAP için bir pencere ve model seçelim
# Genelde full_36_42 veya medium_37_40 en güvenilir olur.
SHAP_WINDOW = "medium_37_40"
SHAP_TEST_CELL = "Cell1"

feature_cols = get_feature_cols_for_window(SHAP_WINDOW)

shap_df = multi_features.copy()
# İstersen outlier'ları çıkar:
# shap_df = shap_df[~shap_df["soh_outlier"]].copy()

train_df = shap_df[shap_df["cell"] != SHAP_TEST_CELL].dropna(subset=["soh"]).copy()
test_df = shap_df[shap_df["cell"] == SHAP_TEST_CELL].dropna(subset=["soh"]).copy()

X_train = train_df[feature_cols]
y_train = train_df["soh"].values
X_test = test_df[feature_cols]
y_test = test_df["soh"].values

# Imputer + RF'i ayrı kuralım ki SHAP'a temiz matrix verelim
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

rf_shap = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    min_samples_leaf=2,
    n_jobs=-1
)
rf_shap.fit(X_train_imp, y_train)
pred_test = rf_shap.predict(X_test_imp)

print(f"SHAP modeli: {SHAP_WINDOW}, test cell: {SHAP_TEST_CELL}")
print("RMSE:", rmse(y_test, pred_test))
print("MAE:", mean_absolute_error(y_test, pred_test))
print("R2:", r2_score(y_test, pred_test))

X_test_imp_df = pd.DataFrame(X_test_imp, columns=feature_cols)

# Çok büyükse örnekle
sample_n = min(300, len(X_test_imp_df))
X_shap = X_test_imp_df.sample(sample_n, random_state=42)

explainer = shap.TreeExplainer(rf_shap)
shap_values = explainer.shap_values(X_shap)

# SHAP summary plot
shap.summary_plot(shap_values, X_shap, show=True)

# Ortalama mutlak SHAP değerleri tablo
shap_importance = pd.DataFrame({
    "feature": feature_cols,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("SHAP önem sıralaması:")
display(shap_importance)

# Bar plot
plt.figure(figsize=(8, 6))
top_shap = shap_importance.head(15).iloc[::-1]
plt.barh(top_shap["feature"], top_shap["mean_abs_shap"])
plt.xlabel("Mean |SHAP value|")
plt.title(f"Top SHAP features - {SHAP_WINDOW}")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 31) SONUÇLARI KAYDET
# ============================================================

RESULT_DIR = os.path.join(BASE_DIR, "multi_hi_partial_results")
os.makedirs(RESULT_DIR, exist_ok=True)

multi_features.to_csv(os.path.join(RESULT_DIR, "multi_hi_features.csv"), index=False)
corr_multi_df.to_csv(os.path.join(RESULT_DIR, "multi_hi_correlation_with_soh.csv"), index=False)
partial_results_df.to_csv(os.path.join(RESULT_DIR, "multi_hi_partial_model_results.csv"), index=False)
single_hi_results_df.to_csv(os.path.join(RESULT_DIR, "single_hi_ekf_partial_results.csv"), index=False)
summary_compare.to_csv(os.path.join(RESULT_DIR, "single_vs_multi_hi_summary_compare.csv"), index=False)
missing_avg.to_csv(os.path.join(RESULT_DIR, "missing_hi_robustness_summary.csv"), index=False)

try:
    shap_importance.to_csv(os.path.join(RESULT_DIR, "shap_importance.csv"), index=False)
except Exception:
    pass

print("Sonuçlar kaydedildi:")
print(RESULT_DIR)

print("\nKaydedilen dosyalar:")
for f in os.listdir(RESULT_DIR):
    print("-", f)

In [ ]:
# ============================================================
# 32) KISA YORUM ŞABLONU
# ============================================================
# Bu hücre sonuçlara bakıp makale notu yazman için otomatik kısa özet üretir.

best_raw = summary_compare[summary_compare["exclude_outliers"] == False].sort_values("RMSE").iloc[0]
best_clean = summary_compare[summary_compare["exclude_outliers"] == True].sort_values("RMSE").iloc[0]

print("RAW veri en iyi sonuç:")
print(best_raw)

print("\nOutlier-aware veri en iyi sonuç:")
print(best_clean)

print("\nYazılabilecek kısa yorum:")
print(f"""
Oxford Battery Degradation Dataset üzerinde yapılan deneylerde, ICA tabanlı tekil normalized Pmax göstergesinin SoH ile güçlü ilişki gösterdiği doğrulanmıştır.
Bunun ötesinde, farklı partial charging pencerelerinden çıkarılan çoklu sağlık göstergeleri kullanılarak Single-HI EKF yaklaşımı ile Multi-HI makine öğrenmesi modelleri karşılaştırılmıştır.

Raw veri üzerinde en iyi ortalama sonuç:
- Window: {best_raw['window']}
- Model: {best_raw['model']}
- RMSE: {best_raw['RMSE']:.5f}
- MAE: {best_raw['MAE']:.5f}
- R2: {best_raw['R2']:.5f}

Outlier-aware veri üzerinde en iyi ortalama sonuç:
- Window: {best_clean['window']}
- Model: {best_clean['model']}
- RMSE: {best_clean['RMSE']:.5f}
- MAE: {best_clean['MAE']:.5f}
- R2: {best_clean['R2']:.5f}

Bu sonuçlar, örnek makaledeki tek normalized Pmax tabanlı yaklaşımın ötesine geçilerek, partial charging koşullarında çıkarılabilen çoklu HI özelliklerinin tahmin başarısı ve dayanıklılık açısından ayrıca değerlendirilmesine imkân sağlamaktadır.
""")
